# ETL & Feature Engineering Pipeline

## Project Overview
This notebook demonstrates the Extract-Transform-Load (ETL) process and Feature Engineering for the Stock Analysis Project. 
The goal is to prepare a high-quality dataset for training our machine learning models (XGBoost).

### Pipeline Steps:
1. **Data Extraction**: Fetch historical stock data using `yfinance`.
2. **Data Cleaning**: Handle missing values and ensure timezone consistency.
3. **Feature Engineering**: Generate technical indicators (RSI, MACD, Bollinger Bands) and integrate sentiment data.
4. **Data Export**: Save the processed feature set for model training.

---

In [ ]:
# Import essential libraries
import yfinance as yf
import pandas as pd
import numpy as np
import talib
import os
import matplotlib.pyplot as plt

# Ensure data directories exist
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Environment Setup Complete.")

## 1. Data Extraction
We fetch 2 years of historical data for a target stock (e.g., RELIANCE.NS). This provides sufficient history for rolling windows and trend analysis.

In [ ]:
SYMBOL = "RELIANCE.NS"
PERIOD = "2y"
INTERVAL = "1d"

def fetch_data(symbol, period, interval):
    print(f"Fetching data for {symbol}...")
    ticker = yf.Ticker(symbol)
    df = ticker.history(period=period, interval=interval)
    
    # Flatten MultiIndex if present (yfinance update)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    # Ensure Date is timezone-naive for consistency
    if df['Date'].dt.tz is not None:
        df['Date'] = df['Date'].dt.tz_localize(None)
        
    return df

raw_df = fetch_data(SYMBOL, PERIOD, INTERVAL)
print(f"Fetched {len(raw_df)} rows.")
raw_df.tail()

## 2. Feature Engineering
We transform raw price data (Open, High, Low, Close, Volume) into predictive features. 
Target features include:
- **Momentum**: RSI (Relative Strength Index)
- **Trend**: SMA (Simple Moving Average), EMA, MACD
- **Volatility**: Bollinger Bands, ATR (Average True Range)
- **Volume**: Volume Change

In [ ]:
def add_technical_indicators(df):
    df = df.copy()
    close = df['Close']
    high = df['High']
    low = df['Low']
    
    # 1. Moving Averages
    df['SMA_20'] = talib.SMA(close, timeperiod=20)
    df['SMA_50'] = talib.SMA(close, timeperiod=50)
    df['EMA_12'] = talib.EMA(close, timeperiod=12)
    
    # 2. RSI (Momentum)
    df['RSI'] = talib.RSI(close, timeperiod=14)
    
    # 3. MACD (Trend)
    macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    df['MACD'] = macd
    df['MACD_signal'] = macdsignal
    
    # 4. Bollinger Bands (Volatility)
    upper, middle, lower = talib.BBANDS(close, timeperiod=20)
    df['BB_upper'] = upper
    df['BB_lower'] = lower
    
    # 5. ATR (Volatility)
    df['ATR'] = talib.ATR(high, low, close, timeperiod=14)
    
    # 6. Volume Change
    df['Volume_Change'] = df['Volume'].pct_change()
    
    # 7. Lagged Features (Previous day's close for prediction)
    df['Prev_Close'] = df['Close'].shift(1)
    df['Daily_Return'] = df['Close'].pct_change()
    
    return df

feature_df = add_technical_indicators(raw_df)
feature_df = feature_df.dropna() # Remove initial rows with NaN due to rolling windows
print(f"Dataset shape after feature engineering: {feature_df.shape}")
feature_df.tail()

## 3. Sentiment Integration (Mock Data for ETL Demo)
In a production environment, we would merge this with News/Twitter sentiment. Here we simulate a 'Sentiment Score' to ensure our model pipeline can handle multi-modal data.

In [ ]:
# Simulate daily sentiment score (0 to 1)
np.random.seed(42)
feature_df['Sentiment_Score'] = np.random.uniform(0.4, 0.8, size=len(feature_df))
feature_df['News_Volume'] = np.random.randint(0, 20, size=len(feature_df))

print("Integrated Sentiment Features.")

## 4. Feature Selection & Target Creation
We define our Target Variable: **10-day future return**.
The model will predict if the stock is a "BUY" based on predicted future return.

In [ ]:
# Create Target: Return 10 days into the future
PREDICTION_HORIZON = 10

feature_df['Future_Close'] = feature_df['Close'].shift(-PREDICTION_HORIZON)
feature_df['Target_Return'] = (feature_df['Future_Close'] - feature_df['Close']) / feature_df['Close']

# Clean NaNs created by shifting
final_df = feature_df.dropna()

print(f"Final Dataset for Training: {final_df.shape}")
print("Columns:", final_df.columns.tolist())

## 5. Save Processed Data
We save this dataset to a CSV/Parquet file, which will be the input for our Model Training notebook.

In [ ]:
OUTPUT_PATH = "data/processed/training_data.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Data successfully saved to {OUTPUT_PATH}")